# Bathroom Monitoring Run — Plots

Plots for the controlled indoor validation run of the acoustic flood detector.
The detector logs one prediction per ~15 s chunk into a CSV with columns:
`timestamp, flood_water_%, rain_%, ambient_dry_%, prediction`.

**How to use in Colab:** run the first cell, then upload `bathroom_sound_segmentation.csv`
when prompted (or set `CSV_PATH` manually if the file is already on the runtime / Drive).

In [ ]:
# --- Setup & load data ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from google.colab import drive
drive.mount('/content/drive')

CSV_PATH = '/content/drive/MyDrive/Python course/DIY/final_project/log1.csv'  # change if needed


df = pd.read_csv(CSV_PATH)

# Strip the '%' sign and convert the probability columns to float
for c in ['flood_water_%', 'rain_%', 'ambient_dry_%']:
    df[c] = df[c].astype(str).str.replace('%', '', regex=False).astype(float)

# dayfirst=True handles dd/mm/yyyy; format='mixed' also parses ISO rows in the same file
df['timestamp'] = pd.to_datetime(df['timestamp'], dayfirst=True, format='mixed')
df = df.sort_values('timestamp').reset_index(drop=True)

# Keep only the continuous run (the main multi-day session).
# Adjust or remove this filter if you want every record.
df = df[df['timestamp'] >= '2026-06-17'].reset_index(drop=True)

print(f'Records: {len(df)}')
print(f'Range:   {df["timestamp"].min()}  ->  {df["timestamp"].max()}')
df.head()

In [ ]:
# --- Summary statistics ---
colors = {'ambient_dry': '#4C9A2A', 'rain': '#2E75B6', 'flood_water': '#C00000'}

total = len(df)
counts = df['prediction'].value_counts()
for cls in ['ambient_dry', 'rain', 'flood_water']:
    n = counts.get(cls, 0)
    print(f'{cls:12s}: {n:5d}  ({100*n/total:5.2f}%)')

fp = (df['prediction'] != 'ambient_dry').sum()
print(f'\nFalse-positive rate (non-ambient): {100*fp/total:.2f}%')

# Consecutive flood-detection run lengths (relevant to the 3-in-a-row alert rule)
runs, cur = [], 0
for p in df['prediction']:
    if p == 'flood_water':
        cur += 1
    else:
        if cur:
            runs.append(cur)
        cur = 0
if cur:
    runs.append(cur)
print(f'flood runs >=3 consecutive: {sum(r>=3 for r in runs)}  |  >=2: {sum(r>=2 for r in runs)}')

## Figure 1 — Class probabilities over time (time series)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


# 3. Create the missing 'run' column
# We define a "new run" whenever there is a logging gap larger than 10 minutes
threshold = pd.Timedelta(minutes=10)
df['run'] = (df['timestamp'].diff() > threshold).cumsum()

# --- Drop tiny stray runs (e.g. a single chunk logged mid-gap) ---
MIN_CHUNKS = 10  # runs shorter than this are noise, not real sessions

run_sizes = df['run'].value_counts()
keep = run_sizes[run_sizes >= MIN_CHUNKS].index
df = df[df['run'].isin(keep)].copy()

# Re-number the surviving runs 0, 1, 2... so panels label cleanly
df['run'] = df['run'].map({old: new for new, old in enumerate(sorted(keep))})

run_ids = sorted(df['run'].unique())
print(f'Kept {len(run_ids)} run(s):')
for r in run_ids:
    sub = df[df['run'] == r]
    print(f'  run {r}: {len(sub):4d} chunks   {sub["timestamp"].min()}  ->  {sub["timestamp"].max()}')

colors = {'ambient_dry': '#4C9A2A', 'rain': '#2E75B6', 'flood_water': '#C00000'}

fig, axes = plt.subplots(1, len(run_ids), figsize=(13, 4.5), sharey=True)
if len(run_ids) == 1:
    axes = [axes]

for ax, r in zip(axes, run_ids):
    sub = df[df['run'] == r]
    x = sub['timestamp']
    ax.plot(x, sub['ambient_dry_%'], color=colors['ambient_dry'], lw=0.7, alpha=0.9, label='ambient_dry')
    ax.plot(x, sub['rain_%'],        color=colors['rain'],        lw=0.7, alpha=0.85, label='rain')
    ax.plot(x, sub['flood_water_%'], color=colors['flood_water'], lw=0.7, alpha=0.85, label='flood_water')

    fp_rows = sub[sub['prediction'] == 'flood_water']
    ax.scatter(fp_rows['timestamp'], fp_rows['flood_water_%'],
               color=colors['flood_water'], s=16, zorder=5, edgecolor='white', linewidth=0.4)

    ax.set_ylim(-2, 103)
    ax.set_xlabel('Time')
    ax.set_title(f'Run {r+1}  (n={len(sub)})')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%H:%M'))
    ax.grid(True, alpha=0.2)
    ax.margins(x=0.02)

axes[0].set_ylabel('Class probability (%)')
axes[0].legend(loc='center left', framealpha=0.9, fontsize=8)
fig.suptitle('Bathroom monitoring run — class probabilities over time (split at logging gap)')
plt.tight_layout()
plt.show()

## Figure 2 — Smoothed time series (rolling mean)
Optional cleaner view for a busy log: a rolling average reduces the spike clutter.

In [ ]:
W = 20  # window size in chunks (~5 min at 15 s/chunk)

fig, axes = plt.subplots(len(run_ids), 1, figsize=(12, 7), sharey=True, sharex=True)
if len(run_ids) == 1:
    axes = [axes]

for ax, r in zip(axes, run_ids):
    sub = df[df['run'] == r].copy()
    # hours elapsed since the start of this run
    sub['elapsed_h'] = (sub['timestamp'] - sub['timestamp'].min()).dt.total_seconds() / 3600
    start = sub['timestamp'].min().strftime('%b %d %H:%M')
    for cls, col in [('ambient_dry_%', colors['ambient_dry']),
                     ('rain_%', colors['rain']),
                     ('flood_water_%', colors['flood_water'])]:
        ax.plot(sub['elapsed_h'], sub[cls].rolling(W, center=True).mean(),
                color=col, lw=1.4, label=cls.replace('_%', ''))

    ax.set_ylim(-2, 103)
    ax.set_ylabel('Class prob. (%)')
    ax.set_title(f'Run {r+1}  (start {start},  n={len(sub)})')
    ax.grid(True, alpha=0.2)
    ax.margins(x=0.01)

axes[-1].set_xlabel('Hours since start of run')
axes[0].legend(loc='center left', framealpha=0.9, fontsize=8, ncol=3)
fig.suptitle(f'Bathroom monitoring run — {W}-chunk rolling mean (aligned by elapsed time)')
plt.tight_layout()
plt.savefig('fig_timeseries_smoothed_runs.png', dpi=150)
plt.show()

## Figure 3 — Predicted class per chunk (event raster)
Shows when each non-ambient prediction occurred across the run.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.4))
ymap = {'ambient_dry': 0, 'rain': 1, 'flood_water': 2}
for cls, y in ymap.items():
    sub = df[df['prediction'] == cls]
    ax.scatter(sub['timestamp'], [y]*len(sub), s=14, color=colors[cls])

ax.set_yticks(list(ymap.values()))
ax.set_yticklabels(list(ymap.keys()))
ax.set_xlabel('Time')
ax.set_title('Predicted class per chunk over the monitoring run')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d  %H:%M'))
ax.margins(x=0.01)
plt.tight_layout()
plt.savefig('fig_timeline.png', dpi=150)
plt.show()

## Figure 4 — Prediction distribution

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
order = ['ambient_dry', 'rain', 'flood_water']
vals = [counts.get(c, 0) for c in order]
bars = ax.bar(order, vals, color=[colors[c] for c in order], alpha=0.85)
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v + max(vals)*0.01, str(v), ha='center', fontsize=10)

ax.set_ylabel('Number of chunks')
ax.set_title(f'Prediction distribution (n={total})')
ax.set_ylim(0, max(vals)*1.1)
plt.tight_layout()
plt.savefig('fig_distribution.png', dpi=150)
plt.show()